This notebook us used to score and evaluate Facilities for Vulnerable Road Users


In [6]:
import geopandas as gpd

# First to map out the Public transit access point design by scoring bus stops

geojson_file = "../RoadUsers/Metro_Bus_Stops.geojson"
bus_stops_gdf = gpd.read_file(geojson_file)

In [7]:
import folium

shelter_stops = bus_stops_gdf[bus_stops_gdf["BSTP_SWK_OBS_TCD"] == "SHL"]

centroid = shelter_stops.geometry.unary_union.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=14)

for _, stop in shelter_stops.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=2,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        tooltip=stop["BSTP_SWK_OBS_TCD"]  # Tooltip shows BSPD value
    ).add_to(m)

m

C:\Users\theod\AppData\Local\Temp\ipykernel_13224\851479277.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = shelter_stops.geometry.unary_union.centroid


In [8]:
bus_stop_signs = bus_stops_gdf[
    (bus_stops_gdf["BSTP_PRK_RST_TCD"] == "NSG") &  # Condition for NSG
    (~bus_stops_gdf.index.isin(shelter_stops.index))  # Not in shelter_stops
]

centroid = bus_stop_signs.geometry.unary_union.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=14)

for _, stop in bus_stop_signs.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=2,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
    ).add_to(m)

m

C:\Users\theod\AppData\Local\Temp\ipykernel_13224\159228634.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = bus_stop_signs.geometry.unary_union.centroid


In [11]:
from folium.plugins import MarkerCluster

# All bus stops are Permanent

roads_clipped = gpd.read_file("../roads_clipped.geojson")
roads_buffered = roads_clipped.copy()
roads_buffered["geometry"] = roads_buffered.geometry.buffer(0.0001)


shelter_stops_near_roads = shelter_stops[
    shelter_stops.geometry.apply(lambda x: roads_buffered.geometry.intersects(x).any())
]

bus_stop_signs_near_roads = bus_stop_signs[
    bus_stop_signs.geometry.apply(lambda x: roads_buffered.geometry.intersects(x).any())
]

centroid = roads_clipped.geometry.unary_union.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=14)

# Add road segments to the map
for _, road in roads_clipped.iterrows():
    folium.GeoJson(road["geometry"], color="black", weight=2, tooltip="Road Segment").add_to(m)

    
for _, stop in bus_stop_signs_near_roads.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=2,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
    ).add_to(m)

for _, stop in shelter_stops_near_roads.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=2,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
    ).add_to(m)
m

C:\Users\theod\AppData\Local\Temp\ipykernel_13224\629828682.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roads_buffered["geometry"] = roads_buffered.geometry.buffer(0.0001)
C:\Users\theod\AppData\Local\Temp\ipykernel_13224\629828682.py:18: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = roads_clipped.geometry.unary_union.centroid


In [ ]:
# For each road segment we need to count the number of bus stops that fall within a 0.0001 buffer to score accordingly
roads_clipped["buffer_geometry"] = roads_clipped.geometry.buffer(0.0001)

def count_nearby_stops(road_buffer, stops_gdf):
    return stops_gdf[stops_gdf.geometry.intersects(road_buffer)].shape[0]

roads_clipped["bus_stop_sign_count"] = roads_clipped["buffer_geometry"].apply(lambda buf: count_nearby_stops(buf, bus_stop_signs))
roads_clipped["shelter_stop_count"] = roads_clipped["buffer_geometry"].apply(lambda buf: count_nearby_stops(buf, shelter_stops))

roads_clipped["PUBLIC TRANSIT SCORE"] = None

for index, row in roads_clipped.iterrows():
    
    #Bus shelter present
    if row["shelter_stop_count"] > 0:
        roads_clipped.at[index, "PUBLIC TRANSIT SCORE"] = 0.75
    
    # Bus stop sign present
    elif row["bus_stop_sign_count"] > 0:
        roads_clipped.at[index, "PUBLIC TRANSIT SCORE"] = 0.5
        
    else: # No bus stops or bus shelters are present
        roads_clipped.at[index, "PUBLIC TRANSIT SCORE"] = 1

min_score = roads_clipped["PUBLIC TRANSIT SCORE"].min()
max_score = roads_clipped["PUBLIC TRANSIT SCORE"].max()

# Normalized to give a better visualization where from 0.5 : 1 becomes 0 to 1
roads_clipped["NORMALIZED_PUBLIC_TRANSIT_SCORE"] = (roads_clipped["PUBLIC TRANSIT SCORE"] - min_score) / (max_score - min_score)

C:\Users\theod\AppData\Local\Temp\ipykernel_13224\3037890473.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roads_clipped["buffer_geometry"] = roads_clipped.geometry.buffer(0.0001)


In [ ]:
import folium
from branca.colormap import linear

# Get the centroid of the road segments for initial map view
centroid = roads_clipped.geometry.unary_union.centroid

# Initialize folium map centered on the centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=14)

# Define a LinearColorMap for the scores
colormap = linear.RdBu_09.scale(0, 1)  # Blue to Red scale from 0 to 1

# Plot the roads with color based on their PUBLIC TRANSIT SCORE
for _, road in roads_clipped.iterrows():
    score = road["NORMALIZED_PUBLIC_TRANSIT_SCORE"]
    color = colormap(score)  # Get the color based on score
    
    # Plot the road with the color corresponding to its score
    folium.GeoJson(
        road['geometry'],
        style_function=lambda x, color=color: {
            'color': color,
            'weight': 3,
            'opacity': 0.7
        },
        tooltip=f"Score: {score:.2f}"
    ).add_to(m)
    
for _, stop in bus_stop_signs_near_roads.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=2,
        color="black",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
    ).add_to(m)

for _, stop in shelter_stops_near_roads.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=2,
        color="green",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
    ).add_to(m)
# Add the colormap legend to the map
colormap.add_to(m)

# Show the map
m


C:\Users\theod\AppData\Local\Temp\ipykernel_27896\3481240935.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = roads_clipped.geometry.unary_union.centroid


Still need to identify locations of lay-by bus stops on the roadway

Continuing to Identify Cycling Infrastructure